# MNIST Classification - Figure 2(a), final 48-model experiment

Yalnızca bu notebook'u Colab'a yükleyin. **Runtime > Change runtime type > GPU**
(varsa T4), ardından **Runtime > Run all**. Drive bağlantısı istendiğinde izin verin.

Tamamlanmış yerel deneyin bilimsel konfigürasyonu sabittir: **seed=42,123,2026;
M=1,2,3,4,5; 10 epoch; Adam lr=0.1; batch=100; 55k/5k/10k; split seed=4509**.
Ensemble, Ensemble + R, Ensemble + AT ve MC Dropout(0.1): toplam **48 model**,
**63 ham sonuç satırı**. Profil seçimi, kısa eğitim/smoke veya hiperparametre araması yoktur.

Bu hazırlık **gerçek Colab testi olarak raporlanmaz**. Yerelde bütün hücreler
eğitim kapalı modda doğrulanır; gerçek GPU eğitimini siz daha sonra çalıştıracaksınız.
Yerel eski sonuç/checkpoint'ler notebook'a dahil değildir.


In [ ]:
import os, sys, importlib.util
from pathlib import Path
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
os.environ['MPLBACKEND'] = 'Agg'
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
sys.dont_write_bytecode = True
try:
    IN_COLAB = importlib.util.find_spec('google.colab') is not None
except (ImportError, ModuleNotFoundError, ValueError):
    IN_COLAB = False
VALIDATION_ONLY = not IN_COLAB or os.environ.get('MNIST_COLAB_VALIDATE_ONLY') == '1'
ALLOW_TRAINING = IN_COLAB and not VALIDATION_ONLY
# This identifies a Drive storage slot, NOT an experiment hyperparameter.
# Keep it unchanged after a disconnect to resume the SAME checkpoints.
RUN_ID = os.environ.get('MNIST_COLAB_RUN_ID', 'figure2a_final_local_48')
import re
if not re.fullmatch(r'[A-Za-z0-9_-]+', RUN_ID):
    raise ValueError('Use a simple RUN_ID containing letters, numbers, underscore or hyphen')
print('Mode:', 'REAL COLAB GPU' if ALLOW_TRAINING else 'LOCAL COMPATIBILITY ONLY - NO TRAINING')
print('Drive resume slot:', RUN_ID)


## Ortam ve CUDA
Colab'ın hazır CUDA destekli PyTorch'u korunur. Yalnız eksik NumPy, Pandas veya
Matplotlib kurulabilir. Windows `.venv`, CUDA wheel'i, torchvision veya sklearn
gerekmez. GPU tahsisi Free hesaplarda garanti değildir; GPU yoksa eğitim başlamaz.


In [ ]:
import subprocess, inspect
missing = [requirement for name, requirement in [
    ('numpy', 'numpy>=1.26,<3'), ('pandas', 'pandas>=2.1,<4'), ('matplotlib', 'matplotlib>=3.8,<4')
] if importlib.util.find_spec(name) is None]
if missing:
    if VALIDATION_ONLY:
        raise RuntimeError(f'Local validation dependencies missing: {missing}')
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
import torch, numpy as np, pandas as pd, matplotlib
if 'weights_only' not in inspect.signature(torch.load).parameters:
    raise RuntimeError('Use a clean current Colab runtime with preinstalled PyTorch')
if ALLOW_TRAINING:
    if not torch.cuda.is_available():
        raise RuntimeError('Select Runtime > Change runtime type > GPU before Run all')
    probe = torch.ones(1, device='cuda')
    assert probe.item() == 1
    del probe
    print('GPU:', torch.cuda.get_device_name(0), '| CUDA:', torch.version.cuda)
else:
    def _no_local_optimizer_step(*args, **kwargs):
        raise RuntimeError('Optimizer steps are forbidden during local Colab preparation')
    torch.optim.Adam.step = _no_local_optimizer_step
print('PyTorch:', torch.__version__, '| Python:', sys.version.split()[0])


## Kaynakları aç ve Drive'a bağlan
Kaynak/data/model/training/metric kodları yerel final run ile byte düzeyinde
aynıdır. Çalışma kopyası `/content` altında yenidir; checkpoint'ler ayrı ve kalıcı
Drive dizinine kaydedilir. Bağlantı kesilirse aynı RUN_ID ile Run all uygulayın.


In [ ]:
import base64, gzip, hashlib, json, tempfile
PAYLOAD = '...'
EXPECTED_BUNDLE_SHA256 = '70b731396f853254be9937dcb34540985fe2235904c7a2959323577da8786138'
packed = base64.b64decode(PAYLOAD, validate=True)
assert hashlib.sha256(packed).hexdigest() == EXPECTED_BUNDLE_SHA256
bundle = json.loads(gzip.decompress(packed))
BASE = Path('/content') if IN_COLAB else Path(tempfile.gettempdir())
WORK = Path(tempfile.mkdtemp(prefix='mnist_figure2a_work_', dir=BASE)).resolve()
for relative, encoded in bundle['assets'].items():
    target = WORK / relative
    if ':' in relative or '\\' in relative or not target.resolve().is_relative_to(WORK):
        raise ValueError('Unsafe embedded source path')
    content = base64.b64decode(encoded, validate=True)
    assert hashlib.sha256(content).hexdigest() == bundle['sha256'][relative]
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(content)
(WORK / 'bundle_manifest.json').write_text(json.dumps(bundle['sha256'], indent=2), encoding='utf-8')
sys.path.insert(0, str(WORK))
if ALLOW_TRAINING:
    from google.colab import drive
    drive.mount('/content/drive')
    PERSISTENT = Path('/content/drive/MyDrive/DeepEnsembles_MNIST_Figure2a') / RUN_ID
else:
    PERSISTENT = WORK / 'local_validation_only'
from colab_runtime import ColabRun
run = ColabRun(WORK, PERSISTENT, ALLOW_TRAINING)
RESULT = run.result
print('Isolated source:', WORK)
print('Persistent result:', RESULT)


## Eğitimsiz doğruluk kontrolleri
Matematik, probability averaging, MC Dropout/BN, FGSM/R, 63 satır/21 özet ve
checkpoint formatı/skip/resume kontrolleri. Test fixture'ları eğitilmemiş yapay
format örnekleridir; gerçek deney sonuçlarına veya checkpoint dizinine konmaz.
Hiçbir optimizer adımı veya kısa model eğitimi çalıştırılmaz.


In [ ]:
run.run_no_training_tests()


## MNIST ve aynı preprocessing / bölme
Gerçek Colab'da resmi IDX arşivleri otomatik indirilip MD5 ile doğrulanır.
Pixel/255, float32, 784 giriş, 55k/5k/10k ve split seed=4509 korunur.
Train/validation/test indeksleri yerel final run'ın array SHA-256'larıyla da karşılaştırılır.
Yerel kontrol yalnız mevcut veri cache'inin geçici kopyasını kullanır; indirme yapmaz.


In [ ]:
local_cache = os.environ.get('MNIST_COLAB_LOCAL_DATA') if VALIDATION_ONLY else None
run.prepare_data(local_cache=local_cache)


## Sabit final deney - 48 model, 10 epoch
Her epoch checkpoint'i doğrudan Drive'a atomik olarak kaydedilir. Seed, yöntem,
üye ve epoch ilerlemesi aşağıda görünür. Geçerli tamamlanmış checkpoint'ler
doğrulanıp atlanır; eksik model son tamamlanan epoch'tan devam eder.
Bozuk/config'i uyumsuz checkpoint silinip yeniden eğitilmez; hata ile durulur.
Kısmi modele devam için aynı PyTorch/CUDA/GPU ortamı gerekir.


In [ ]:
run.run_final()


In [ ]:
from IPython.display import display, Image
if ALLOW_TRAINING:
    manifest = json.loads((RESULT / 'manifest.json').read_text(encoding='utf-8'))
    assert manifest['status'] == 'complete'
    assert manifest['trained_models'] == 48 and manifest['result_rows'] == 63
    display(pd.read_csv(RESULT / 'summary.csv'))
    display(Image(filename=str(RESULT / 'figures/figure2a_comparison.png'), width=1200))
    print('Total completed epoch training seconds:', manifest['total_training_epoch_seconds'])
    print('Hardware/environment:', manifest['environment'])
else:
    print('Local check only: no trained models or classification results are claimed')


## Tam ZIP: checkpoint'ler ve sonraki OOD deneyi
Bu ZIP yalnız rapor değildir: **48 checkpoint**, config, manifest, 63 sonuç,
21 mean±sample SD özeti, PNG/PDF'ler, kaynaklar ve resmi MNIST arşivlerini içerir.
Checkpoint şeması, MLP state_dict anahtarları ve `outputs/local/checkpoints/seed_*/...`
yerleşimi değişmez; sonraki MNIST + NotMNIST OOD deneyi dönüştürmeden kullanabilir.
ZIP benzersiz bir üst klasör içerir; eski yerel sonuçların üstüne açılmaz.
Drive kopyası kalıcıdır; tarayıcı indirmesi engellenirse Files/Drive'dan ZIP'i indirin.


In [ ]:
ARCHIVE = run.export()
if ARCHIVE is not None:
    print('Complete checkpoint ZIP:', ARCHIVE)
    print('SHA-256:', run.report['export']['sha256'])
    from google.colab import files
    files.download(str(ARCHIVE))
else:
    print('No real Colab ZIP produced in local validation mode')
print('Session manifest:', PERSISTENT / 'colab_session.json')
